In [2]:
import requests
import os
from dotenv import load_dotenv
import xml.etree.ElementTree as ET
import pandas as pd
from datetime import datetime, time
from zoneinfo import ZoneInfo

In [3]:
load_dotenv()

True

In [4]:
api_key = os.getenv("ENTSOE_API_KEY")

In [5]:
payload={}
headers = {
  "SECURITY_TOKEN": api_key
}

In [6]:
def convert_local_midnight_to_utc(date_str):
    pure_date = datetime.strptime(date_str, "%Y-%m-%d").date()
    prague_tz = ZoneInfo("Europe/Prague")
    local_midnight = datetime.combine(pure_date, time.min, tzinfo=prague_tz)
    utc_dt = local_midnight.astimezone(ZoneInfo("UTC"))
    
    # ENTSO-E needs YYYYMMDDHHmm
    return utc_dt.strftime("%Y%m%d%H%M")

In [7]:
target_types = {
    "B01": "czechia_biomass_gen_last_week",
    "B02": "czechia_lignite_gen_last_week",
    "B04": "czechia_fossil_gas_gen_last_week",
    "B05": "czechia_hard_coal_gen_last_week",
    "B10": "czechia_hydro_active_gen_last_week",
    "B11": "czechia_hydro_passive_gen_last_week",
    "B12": "czechia_hydro_reservoir_gen_last_week",
    "B14": "czechia_nuclear_gen_last_week",
    "B20": "czechia_other_gen_last_week"
}

In [8]:
def parse_xml_response(xml_string):
    root = ET.fromstring(xml_string)

    ns = {'gmd': 'urn:iec62325.351:tc57wg16:451-6:generationloaddocument:3:0'}
    
    raw_production_data = {code: [] for code in target_types.keys()}
    global_max_position = 0
    
    for timeseries in root.findall('gmd:TimeSeries', ns):
        psr_type_elem = timeseries.find('.//gmd:psrType', ns)
        
        if psr_type_elem is not None and psr_type_elem.text in target_types:
            psr_type_code = psr_type_elem.text
            
            for point in timeseries.findall('.//gmd:Point', ns):
                pos_elem = point.find('gmd:position', ns)
                qty_elem = point.find('gmd:quantity', ns)
                
                if pos_elem is not None and qty_elem is not None:
                    pos_val = int(pos_elem.text)
                    
                    # Update our global maximum position tracker
                    if pos_val > global_max_position:
                        global_max_position = pos_val
                        
                    raw_production_data[psr_type_code].append({
                        "pos": pos_val,
                        "gen": float(qty_elem.text)
                    })
    
    processed_data = {}
    
    if global_max_position > 0:
        for code, array in raw_production_data.items():
            if not array:
                processed_data[code] = [{"pos": i, "gen": 0.0} for i in range(1, global_max_position + 1)]
                continue
                
            df = pd.DataFrame(array)
            df = df.groupby('pos').sum()
            df = df.reindex(range(1, global_max_position + 1))
            df['gen'] = df['gen'].ffill()
            df['gen'] = df['gen'].bfill().fillna(0)
            
            df.reset_index(inplace=True)
            processed_data[code] = df.to_dict(orient='records')
            
    return processed_data

In [9]:
df_hourly = pd.DataFrame()

In [76]:
# data for 2022
url = f"https://web-api.tp.entsoe.eu/api?documentType=A75&processType=A16&in_Domain=10YCZ-CEPS-----N&periodStart={convert_local_midnight_to_utc('2022-12-25')}&periodEnd={convert_local_midnight_to_utc('2023-01-01')}"
response = requests.request("GET", url, headers=headers, data=payload)

parsed_data = parse_xml_response(response.text)
df_temp = pd.DataFrame()
for k in target_types:
    df_temp[target_types[k]] = [r['gen'] for r in parsed_data[k]]

print(len(df_temp))

df_hourly = pd.concat([df_hourly, df_temp], ignore_index=True)

168


In [77]:
# data for 2023
url = f"https://web-api.tp.entsoe.eu/api?documentType=A75&processType=A16&in_Domain=10YCZ-CEPS-----N&periodStart={convert_local_midnight_to_utc('2023-01-01')}&periodEnd={convert_local_midnight_to_utc('2024-01-01')}"
response = requests.request("GET", url, headers=headers, data=payload)

parsed_data = parse_xml_response(response.text)
df_temp = pd.DataFrame()
for k in target_types:
    df_temp[target_types[k]] = [r['gen'] for r in parsed_data[k]]

print(len(df_temp))

df_hourly = pd.concat([df_hourly, df_temp], ignore_index=True)

8760


In [78]:
# data for 2024 until 2024-07-01
url = f"https://web-api.tp.entsoe.eu/api?documentType=A75&processType=A16&in_Domain=10YCZ-CEPS-----N&periodStart={convert_local_midnight_to_utc('2024-01-01')}&periodEnd={convert_local_midnight_to_utc('2024-06-30')}"
response = requests.request("GET", url, headers=headers, data=payload)

parsed_data = parse_xml_response(response.text)
df_temp = pd.DataFrame()
for k in target_types:
    df_temp[target_types[k]] = [r['gen'] for r in parsed_data[k]]

print(len(df_temp))

df_hourly = pd.concat([df_hourly, df_temp], ignore_index=True)

4343


In [79]:
czechia_biomass_gen_last_week = [
    189.68, 184.97, 183.73, 187.67, 180.19, 177.20, 180.33, 180.10, 179.69, 
    180.11, 169.53, 166.30, 166.86, 169.80, 179.00, 165.79, 166.53, 168.18, 
    180.52, 179.93, 180.84, 181.16, 170.01, 179.90
]

czechia_lignite_gen_last_week = [
    934.28, 786.05, 770.46, 787.82, 756.53, 786.89, 782.84, 735.99, 757.84, 
    775.22, 756.55, 712.84, 709.33, 717.18, 704.99, 733.56, 860.99, 1111.83, 
    1335.50, 1545.89, 1620.51, 1650.81, 1661.70, 1648.12
]

czechia_fossil_gas_gen_last_week = [
    89.73, 89.54, 89.33, 89.41, 89.47, 91.05, 97.39, 95.79, 94.53, 
    92.70, 90.58, 88.54, 88.33, 88.40, 87.76, 88.10, 89.08, 91.06, 
    93.20, 95.75, 95.50, 96.18, 90.12, 90.49
]

czechia_hard_coal_gen_last_week = [
    2.09, 0.77, 0.53, 0.55, 0.00, 6.27, 7.37, 0.00, 0.00, 
    1.49, 0.62, 0.45, 0.00, 0.00, 0.00, 0.00, 1.15, 1.73, 
    0.13, 0.00, 0.06, 3.45, 1.04, 0.00
]

czechia_hydro_active_gen_last_week = [
    194.29, 82.10, 39.35, 0.00, 0.00, 0.00, 0.00, 37.18, 14.79, 
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 
    494.83, 488.70, 401.36, 511.96, 607.88, 229.84
]

czechia_hydro_passive_gen_last_week = [
    112.49, 112.50, 111.36, 112.61, 111.99, 114.32, 115.70, 113.39, 111.97, 
    107.90, 109.99, 106.96, 111.12, 113.38, 113.38, 110.39, 112.26, 110.44, 
    109.28, 107.93, 93.43, 94.36, 95.55, 103.88
]

czechia_hydro_reservoir_gen_last_week = [
    18.94, 18.89, 25.93, 18.28, 22.00, 28.84, 27.28, 35.50, 22.07, 
    19.46, 14.08, 18.38, 14.04, 13.89, 13.91, 18.10, 13.87, 17.77, 
    18.02, 34.02, 399.66, 421.14, 415.30, 138.88
]

czechia_nuclear_gen_last_week = [
    3363.55, 3365.89, 3357.29, 3355.62, 3370.94, 3374.88, 3379.73, 3336.85, 3334.80, 
    3364.78, 3370.92, 3369.52, 3367.97, 3367.56, 3363.58, 3360.89, 3378.58, 3388.04, 
    3342.71, 3338.09, 3374.50, 3400.25, 3403.66, 3405.09
]

czechia_other_gen_last_week = [
    66.09, 65.79, 65.45, 65.63, 65.64, 67.17, 77.89, 74.82, 73.16, 
    70.62, 67.52, 63.88, 63.65, 63.98, 63.11, 63.36, 64.60, 66.07, 
    70.23, 74.57, 76.59, 77.40, 66.32, 66.60
]

data = {
    "czechia_biomass_gen_last_week": czechia_biomass_gen_last_week,
    "czechia_lignite_gen_last_week": czechia_lignite_gen_last_week,
    "czechia_fossil_gas_gen_last_week": czechia_fossil_gas_gen_last_week,
    "czechia_hard_coal_gen_last_week": czechia_hard_coal_gen_last_week,
    "czechia_hydro_active_gen_last_week": czechia_hydro_active_gen_last_week,
    "czechia_hydro_passive_gen_last_week": czechia_hydro_passive_gen_last_week,
    "czechia_hydro_reservoir_gen_last_week": czechia_hydro_reservoir_gen_last_week,
    "czechia_nuclear_gen_last_week": czechia_nuclear_gen_last_week,
    "czechia_other_gen_last_week": czechia_other_gen_last_week
}

df_temp = pd.DataFrame(data)
df_hourly = pd.concat([df_hourly, df_temp], ignore_index=True)

In [80]:
df_quarterly = df_hourly.loc[df_hourly.index.repeat(4)].reset_index(drop=True)
print(len(df_quarterly))

53180


In [81]:
# data for the rest of 2024
url = f"https://web-api.tp.entsoe.eu/api?documentType=A75&processType=A16&in_Domain=10YCZ-CEPS-----N&periodStart={convert_local_midnight_to_utc('2024-07-01')}&periodEnd={convert_local_midnight_to_utc('2025-01-01')}"
response = requests.request("GET", url, headers=headers, data=payload)

parsed_data = parse_xml_response(response.text)
df_temp = pd.DataFrame()
for k in target_types:
    df_temp[target_types[k]] = [r['gen'] for r in parsed_data[k]]

print(len(df_temp))

df_quarterly = pd.concat([df_quarterly, df_temp], ignore_index=True)

17668


In [13]:
# data 2025
url = f"https://web-api.tp.entsoe.eu/api?documentType=A75&processType=A16&in_Domain=10YCZ-CEPS-----N&periodStart={convert_local_midnight_to_utc('2025-01-01')}&periodEnd={convert_local_midnight_to_utc('2025-12-25')}"
response = requests.request("GET", url, headers=headers, data=payload)

parsed_data = parse_xml_response(response.text)
df_temp = pd.DataFrame()
for k in target_types:
    df_temp[target_types[k]] = [r['gen'] for r in parsed_data[k]]

print(len(df_temp))

df_quarterly = pd.concat([df_quarterly, df_temp], ignore_index=True)

5184


In [84]:
df_quarterly.to_csv("../data/czechia_other_sources_generation_last_week_15min.csv", sep=";", index=False)